In [18]:
import pandas as pd
import mhcgnomes
pd.set_option("display.max_columns",500)
PROTEIN_CHECK="^[ACDEFGHIKLMNPQRSTVWY]+$"
HOST_SPECIES = ['human', 'mouse']
CDR3_CHAINS = ['alpha', 'beta']

In [4]:
raw_data = pd.read_csv("../../data/raw-data/McPAS/McPAS.csv",sep = ';')
raw_data.head()

/scratch/ipykernel_2923216/1572331163.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv("../../data/raw-data/McPAS/McPAS.csv",sep = ';')


,CDR3.alpha.aa,CDR3.beta.aa,Species,Category,Pathology,Pathology.Mesh.ID,Additional.study.details,Antigen.identification.method,Single.cell,NGS,Antigen.protein,Protein.ID,Epitope.peptide,Epitope.ID,MHC,Tissue,T.Cell.Type,T.cell.characteristics,CDR3.alpha.nt,TRAV,TRAJ,TRBV,TRBD,TRBJ,Reconstructed.J.annotation,CDR3.beta.nt,Mouse.strain,PubMed.ID,Remarks
0,NaN,ATSIRFTDTQYF,Human,Autoimmune,Celiac disease,D002503,6 days post oral gluten\nchallenge,1.0,Yes,Yes,DQ2-a-II,NaN,PQPELPYPQPQ,NaN,HLA-DQ2,PBMC,CD4,NaN,NaN,TRAV8-6,NaN,TRBV7-2,NaN,TRBJ2-3,NaN,NaN,NaN,23878218,NaN
1,NaN,CAAAAASGAYEQYF,Human,Pathogens,M.Tuberculosis,D009169,Bulk,2.2,No,Yes,NaN,NaN,NaN,NaN,NaN,PBMC,CD4,NaN,NaN,NaN,NaN,TRBV10-3,NaN,NaN,NaN,NaN,NaN,32341563,NaN
2,NaN,CAAADEEIGNQPQHF,Human,Pathogens,Hepatitis C virus,D016174,NaN,1.0,No,No,Genome polyprotein,P27958,ATDALMTGY,4917,HLA-A*01:01,PBMC,CD8,NaN,NaN,NaN,NaN,TRBV10-03,NaN,NaN,No,NaN,NaN,21160049,NaN
3,CAVRDSNYQLIW,CAAAGGPDTGELFF,Human,Pathogens,M. tuberculosis,D009169,NaN,2.2,Yes,Yes,NaN,NaN,NaN,NaN,NaN,PBMC,CD8,MAIT,NaN,NaN,TRAJ33,TRBV6-1,NaN,TRBJ2-2,NaN,NaN,NaN,30992377,NaN
4,NaN,CAAEDDTGGFKTIF,Human,Pathogens,Influenza,D009980,NaN,1.0,Yes,Yes,Matrix protein (M1),P03485,GILGFVFTL,20354,HLA-A*02:01,PBMC,CD8,NaN,NaN,TRAV8-4:03,TRAJ45:01,TRBV25:01,NaN,TRBJ9:01,No,NaN,NaN,28300170,NaN


In [6]:
raw_data["MHC"].unique()

array(['HLA-DQ2', nan, 'HLA-A*01:01', 'HLA-A*02:01', 'HLA-A*02',
       'HLA-B*57:01', 'HLA-A2:01', 'HLA-B*07:02', 'HLA-DRB1', 'H-2bxH-2z',
       'HLA-C*07:02', 'HLA-B*57', 'DRB1*04-01', 'H-2Db', 'HLA-A2', 'H-2b',
       'H-2Kb', 'HLA-B7', 'HLA-A*11:01', 'HLA-A1', 'HLA-B*08:01',
       'HLA-A*24:02', 'H-2k', 'Â\xa0HLA-DQB1*06:02', 'HLA-B*42:01',
       'H-2Kd', 'HLA-B*57:03', 'HLA-B*08', 'HLA-B*35:01',
       'HLA-DRB1*04:01', 'H-2Cg', 'HLA-B*15', 'HLA-B*27:05', 'HLA-DR15',
       'HLA-DQ8', 'H-2q', 'HLA-DR5', 'H-2u', 'HLA-A*2:01', 'H-2d/q',
       'H-2g7', 'DR4-H2-E', 'H-2j', 'HLA-DR4', 'H-2d', 'I-Ab', 'HLA-DR11',
       'HLA-DR1',
       'Complete HLA typing is provided in the paper for each patient',
       'H-2d/b', 'HLA-Cw* 16:01', 'H-2DbÂ\xa0', 'HLA-B*35:02', 'H2-b',
       'HLA-B*27', 'DRB1*04:01', 'HLA-B*14', 'HLA-A*01', 'HLA-B*44:05',
       'H-2db', 'HLA-A*011', 'DRB1*15:03', 'H-2kb', 'HLA-B*8',
       'HLA-A*02:14', 'HLA-A*02:15', 'HLA-A*02:12', 'HLA-DQ2.5',
       'HLA-A*0

In [7]:
raw_data["Species"].unique()

array(['Human', 'Mouse', nan], dtype=object)

In [8]:
raw_data["Remarks"].unique()

array([nan, 'No final F', 'No first Cysteine',
       'No first Cysteine, No final F',
       'TRAV4L-23 and TRAJ4L-23 are "TRAV and TRAJ are homologous but not equivalent to known germline sequences',
       'Reconstructed V', 'Reconstruted J', 'Epitope variannt',
       'verify VJ', 'Stop codon',
       'Brackets represent optional amino acid',
       'No first Cysteine, No final F,  Short Sequence',
       'TRBJ not suplied'], dtype=object)

In [9]:
def calculate_receptor_id(tbl: pd.DataFrame, column_id: str):
    data = tbl.copy(deep=True)
    #receptor data has already wide format
    for i in data.index:
        data.loc[i,"id"] = str(uuid.uuid4())

    return data

In [13]:
def fix_mhc_name(allele_str, chain = 'alpha'):
    try:
        # Предварительные проверки и подготовления
        if pd.isna(allele_str):
            return pd.NA     
        allele_first = allele_str.split(" ")[0]
        if allele_first == "B2M":
            return "B2M"
        # Получение конкретной аллели
        parsed_allele = mhcgnomes.parse(allele_first)
        if isinstance(parsed_allele, mhcgnomes.pair.Pair):
            if chain == 'alpha':
                allele = parsed_allele.alpha
            elif chain == 'beta':
                allele = parsed_allele.beta
            else:
                raise ValueError('Unknown chain')
        elif isinstance(parsed_allele, mhcgnomes.allele.Allele):
            allele = parsed_allele
        else:
            raise ValueError('Not allele')
        # Проверка
        if allele.gene.species.name == "Homo sapiens":
            if len(allele.allele_fields) < 2:
                raise ValueError('Too many allele fields. Need at least 2.')
            elif len(allele.allele_fields) == 2:
                return allele.to_string()
            else:
                return allele.restrict_allele_fields(2, drop_annotations=True, drop_mutations=True).to_string()
        elif allele.gene.species.name == "Mus musculus":
            return allele.to_string()
        else:
            raise ValueError('Wrong species. Only human and mouse are allowed.')
    except (mhcgnomes.errors.ParseError, TypeError, ValueError):
        return pd.NA


In [11]:
raw_data = pd.read_csv("../../data/raw-data/McPAS/McPAS.csv", sep = ";", header = 0).query("Species.notna()")
print(raw_data.shape)

(40175, 29)


/scratch/ipykernel_2923216/4025321698.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv("../../data/raw-data/McPAS/McPAS.csv", sep = ";", header = 0).query("Species.notna()")


In [16]:
print(raw_data.shape)
print("Unificate species")
raw_data.loc[raw_data["Species"].str.contains("Human"),"Species"] = "human"
raw_data.loc[raw_data["Species"].str.contains("Mouse"),"Species"] = "mouse"
    
raw_data['valid_ii_name'] = "B2M" # not valid mhcii, they will be filtered
raw_data['valid_i_name'] = raw_data['MHC'].apply(lambda x: fix_mhc_name(x, 'alpha'))
raw_data["database"] = "McPAS"
    
table_schema = {
            "id": "id",
            "CDR3.alpha.aa": "cdr3_alpha",
            "CDR3.beta.aa": "cdr3_beta",
            "Epitope.peptide": "epitope",
            "valid_i_name": "mhc_alpha",
            "valid_ii_name": "mhc_beta",
            "mhc_class": "mhc_class",
            "Species": "host_species",
            "Pathology": "epitope_species",
            "Antigen.protein": "epitope_source",
            "TRAV": "V_alpha",
            "TRBV": "V_beta",
            "TRBD": "D_beta",
            "TRAJ": "J_alpha",
            "TRBJ": "J_beta", 
            "database": "database"
        }
    

(40175, 30)
Unificate species


In [39]:
raw_data['valid_cdr3'] = raw_data['CDR3.alpha.aa'].fillna('').str.contains(PROTEIN_CHECK) | raw_data['CDR3.beta.aa'].fillna('').str.contains(PROTEIN_CHECK)
raw_data['valid_cdr3']

0        True
1        True
2        True
3        True
4        True
         ... 
40726    True
40727    True
40728    True
40729    True
40730    True
Name: valid_cdr3, Length: 40175, dtype: bool

In [38]:
raw_data[['CDR3.alpha.aa','CDR3.beta.aa']]

,CDR3.alpha.aa,CDR3.beta.aa
0,NaN,ATSIRFTDTQYF
1,NaN,CAAAAASGAYEQYF
2,NaN,CAAADEEIGNQPQHF
3,CAVRDSNYQLIW,CAAAGGPDTGELFF
4,NaN,CAAEDDTGGFKTIF
...,...,...
40726,NaN,CASSPKRTRQLSIS
40727,NaN,CASSPTRPAARRLS
40728,NaN,CASSTRQLFTSVPP
40729,NaN,CASSTPPAARRLGP


In [30]:
raw_data['CDR3.alpha.aa'].str.contains(PROTEIN_CHECK)

0         NaN
1         NaN
2         NaN
3        True
4         NaN
         ... 
40726     NaN
40727     NaN
40728     NaN
40729     NaN
40730     NaN
Name: CDR3.alpha.aa, Length: 40175, dtype: object

In [31]:
raw_data['CDR3.beta.aa'].str.contains(PROTEIN_CHECK)

0        True
1        True
2        True
3        True
4        True
         ... 
40726    True
40727    True
40728    True
40729    True
40730    True
Name: CDR3.beta.aa, Length: 40175, dtype: object

In [37]:
import numpy as np
bool(np.nan) | True

True

In [43]:
print("Apply filters")
clean_data = raw_data.query("`PubMed.ID`.notna()").query("Species.isin(@HOST_SPECIES)").\
query("`CDR3.alpha.aa`.fillna('').str.contains(@PROTEIN_CHECK) or `CDR3.beta.aa`.fillna('').str.contains(@PROTEIN_CHECK)").\
query("`Epitope.peptide`.notna()").query("`Epitope.peptide`.str.contains(@PROTEIN_CHECK)").query("valid_i_name.notna() and valid_ii_name.notna()")
          #  query("`PubMed.ID`.notna()").\
          #  query("Species.isin(@HOST_SPECIES)").\
          #  query("`CDR3.alpha.aa`.str.contains(@PROTEIN_CHECK) or `CDR3.beta.aa`.str.contains(@PROTEIN_CHECK)").\
          #  query("`Epitope.peptide`.notna()").\
          #  query("`Epitope.peptide`.str.contains(@PROTEIN_CHECK)").\
          #  query("valid_i_name.notna() and valid_ii_name.notna()")
print(clean_data.shape)    

Apply filters
(6600, 33)


In [44]:
clean_data['mhc_class'] = clean_data['valid_i_name'].apply(lambda x: 'I' if mhcgnomes.parse(x).is_class1 else 'II')
clean_data = clean_data.query("mhc_class == 'I'") # no valid mhcII entries
print(clean_data.shape)
print("Get id")
clean_data_with_id = calculate_receptor_id(clean_data,"receptor_id")
clean_data_selected = clean_data_with_id.filter(items = list(table_schema.keys()), axis = 1).rename(columns = table_schema)
clean_data_selected["D_alpha"] = pd.NA

(6551, 34)
Get id


NameError: name 'uuid' is not defined

In [46]:
mhcgnomes.parse("B2M").species

Species(name='Homo sapiens', mhc_prefix='HLA')

In [52]:
mhcgnomes.parse("H2-Kk")

Allele(gene=Gene(species=Species(name='Mus musculus', mhc_prefix='H2'), name='K', mutations=()), allele_fields=('k',), annotations=(), mutations=())